# P2.5 · Ước lượng SoC bằng EKF
ECM 1-RC dùng cho nghiên cứu trên các cửa sổ DST/FUDS cấp một cell. Bảng dưới đối chiếu kết quả với reference từ bộ đếm Arbin. Quyết định G10 vẫn là NEEDS_REVIEW và ER4 chưa đánh giá.

Notebook đọc các kết quả đã lưu. Đặt P25_RERUN=1 và P22_ZIP để chạy lại quy trình sau khi P2.4 đã đạt và hồ sơ phê duyệt hợp lệ.

In [1]:
import os, sys
from pathlib import Path
root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
from run_ekf import read_json, run_development
out = root / 'results/p25'
if os.environ.get('P25_RERUN') == '1':
    run_development(Path(os.environ['P22_ZIP']))
report = read_json(out / 'metrics.json')
config = read_json(out / 'ekf_config.json')
print(report['milestone'], report['gate'], config['selected'], report['ER4'])

P2.5 PASS {'id': 'grid-04', 'p0_diag': [0.0016, 0.0004], 'q_per_s_diag': [1e-08, 1e-05], 'r_v2': 0.0036} NOT_EVALUATED


In [2]:
print('Run | population | coverage | MAE | RMSE | P95 | MAX (pp)')
for run, scenarios in report['runs'].items():
    for population in ('FULL_DYNAMIC','NON_FLAGGED','FLAGGED'):
        m = scenarios['nominal']['EKF'][population]
        print(run, population, f"{m['numerator']}/{m['denominator']}", *[m[k] for k in ('MAE_pp','RMSE_pp','P95_abs_pp','MAX_abs_pp')], sep=' | ')

Run | population | coverage | MAE | RMSE | P95 | MAX (pp)
11_05_2015_SP20-2_DST_50SOC | FULL_DYNAMIC | 6698/6698 | 0.6195121494128352 | 0.7702535975510015 | 1.327635874022077 | 1.7614338942376628
11_05_2015_SP20-2_DST_50SOC | NON_FLAGGED | 6415/6415 | 0.6325999070531783 | 0.7823463457010357 | 1.3329460528503485 | 1.7614338942376628
11_05_2015_SP20-2_DST_50SOC | FLAGGED | 283/283 | 0.32284089406725147 | 0.40950859329008915 | 0.8885774356111609 | 1.0101204834629915
11_05_2015_SP20-2_DST_80SOC | FULL_DYNAMIC | 10645/10645 | 0.8550533392179472 | 0.9665987871075817 | 1.7675788119883815 | 1.8883629676201497
11_05_2015_SP20-2_DST_80SOC | NON_FLAGGED | 10369/10369 | 0.8640798838834992 | 0.974724834918514 | 1.773780828240048 | 1.8883629676201497
11_05_2015_SP20-2_DST_80SOC | FLAGGED | 276/276 | 0.5159365216921934 | 0.5845116884094769 | 0.9601012031954168 | 1.0220863125709843
11_06_2015_SP20-2_FUDS_80SOC | FULL_DYNAMIC | 11098/11098 | 0.6149100897103291 | 0.7432003852908909 | 1.3439885393738604 

In [3]:
print('Run | scenario | EKF RMSE (pp) | CC RMSE (pp) | EKF convergence (s)')
for run, scenarios in report['runs'].items():
    for scenario, m in scenarios.items():
        if m['status'] == 'EXECUTED':
            print(run, scenario, m['EKF']['FULL_DYNAMIC']['RMSE_pp'], m['CC']['FULL_DYNAMIC']['RMSE_pp'], m['EKF_convergence_s'], sep=' | ')
        else:
            print(run, scenario, m['reason'], sep=' | ')

Run | scenario | EKF RMSE (pp) | CC RMSE (pp) | EKF convergence (s)
11_05_2015_SP20-2_DST_50SOC | nominal | 0.7702535975510015 | 0.14478357336763148 | 0.0
11_05_2015_SP20-2_DST_50SOC | prior_minus_20pp | 1.290224848079666 | 20.125146070903416 | 63.54640345089501
11_05_2015_SP20-2_DST_50SOC | prior_minus_10pp | 0.769963683560603 | 10.125276950738586 | 17.124413414148876
11_05_2015_SP20-2_DST_50SOC | prior_plus_10pp | 0.9610893083800504 | 9.87525651500132 | 5.046792789747997
11_05_2015_SP20-2_DST_50SOC | prior_plus_20pp | 1.1748883429268924 | 19.875120633370628 | 24.171623329977592
11_05_2015_SP20-2_DST_50SOC | current_bias_minus | 0.5088500949657444 | 2.523471308132068 | 0.0
11_05_2015_SP20-2_DST_50SOC | current_bias_plus | 0.9727178160536243 | 2.80677551125844 | 0.0
11_05_2015_SP20-2_DST_50SOC | vi_noise | 0.7870789158260079 | 0.14948422488103202 | 0.0
11_05_2015_SP20-2_DST_80SOC | nominal | 0.9665987871075817 | 0.07382642503361828 | 0.0
11_05_2015_SP20-2_DST_80SOC | prior_minus_20pp |